In [1]:
import numpy as np
import pandas as pd


In [2]:
credit=pd.read_csv("tmdb_5000_credits.csv")
movie=pd.read_csv("tmdb_5000_movies.csv")

In [3]:
movie=movie[['genres','id','overview','title','keywords']]

In [4]:
movie.isnull().sum()
movie.dropna(inplace=True)

In [5]:
import ast

def convert(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [6]:
movie['genres']=movie['genres'].apply(convert)
movie['keywords']=movie['keywords'].apply(convert)
movie.head()

,genres,id,overview,title,keywords
0,"[Action, Adventure, Fantasy, Science Fiction]",19995,"In the 22nd century, a paraplegic Marine is di...",Avatar,"[culture clash, future, space war, space colon..."
1,"[Adventure, Fantasy, Action]",285,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,"[ocean, drug abuse, exotic island, east india ..."
2,"[Action, Adventure, Crime]",206647,A cryptic message from Bond’s past sends him o...,Spectre,"[spy, based on novel, secret agent, sequel, mi..."
3,"[Action, Crime, Drama, Thriller]",49026,Following the death of District Attorney Harve...,The Dark Knight Rises,"[dc comics, crime fighter, terrorist, secret i..."
4,"[Action, Adventure, Science Fiction]",49529,"John Carter is a war-weary, former military ca...",John Carter,"[based on novel, mars, medallion, space travel..."


In [7]:
movie['overview'] = movie['overview'].fillna('').astype(str).apply(lambda x: x.split())
movie['genres']=movie['genres'].apply(lambda x:[i.replace(" ","") for i in x ])
movie['keywords']=movie['keywords'].apply(lambda x:[i.replace(" ","") for i in x ])
movie.head()

,genres,id,overview,title,keywords
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[cultureclash, future, spacewar, spacecolony, ..."
1,"[Adventure, Fantasy, Action]",285,"[Captain, Barbossa,, long, believed, to, be, d...",Pirates of the Caribbean: At World's End,"[ocean, drugabuse, exoticisland, eastindiatrad..."
2,"[Action, Adventure, Crime]",206647,"[A, cryptic, message, from, Bond’s, past, send...",Spectre,"[spy, basedonnovel, secretagent, sequel, mi6, ..."
3,"[Action, Crime, Drama, Thriller]",49026,"[Following, the, death, of, District, Attorney...",The Dark Knight Rises,"[dccomics, crimefighter, terrorist, secretiden..."
4,"[Action, Adventure, ScienceFiction]",49529,"[John, Carter, is, a, war-weary,, former, mili...",John Carter,"[basedonnovel, mars, medallion, spacetravel, p..."


In [8]:
movie['tags']=movie['overview']+movie['genres']+movie['keywords']

In [9]:
df=movie[['tags','title','id']]
df['tags']=df['tags'].apply(lambda x:" ".join(x))
df.head()

/var/folders/xj/x_jltph909q6cbvm6k4ymp6w0000gn/T/ipykernel_54535/2140744774.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags']=df['tags'].apply(lambda x:" ".join(x))


,tags,title,id
0,"In the 22nd century, a paraplegic Marine is di...",Avatar,19995
1,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,285
2,A cryptic message from Bond’s past sends him o...,Spectre,206647
3,Following the death of District Attorney Harve...,The Dark Knight Rises,49026
4,"John Carter is a war-weary, former military ca...",John Carter,49529


In [10]:
df['tags']=df['tags'].apply(lambda x:x.lower())
df['tags'][0]

/var/folders/xj/x_jltph909q6cbvm6k4ymp6w0000gn/T/ipykernel_54535/1603337806.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags']=df['tags'].apply(lambda x:x.lower())


'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d'

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')


In [12]:
vector=cv.fit_transform(df['tags']).toarray()


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
similar=cosine_similarity(vector)

In [14]:
sorted(list(enumerate(similar[0])), reverse=True, key=lambda x: x[1])[1:6]

[(539, np.float64(0.2732295333235123)),
 (1191, np.float64(0.264575131106459)),
 (507, np.float64(0.25539990311691463)),
 (1213, np.float64(0.2480694691784169)),
 (260, np.float64(0.24688535993934702))]

In [15]:
def rec(movie):
    movie_index=df[df['title']==movie].index[0]
    distance=similar[movie_index]
    movie_list=sorted(list(enumerate(similar[0])), reverse=True, key=lambda x: x[1])[1:6]
    
    
    for i in movie_list:
        print(df.iloc[i[0]].title)

In [16]:
rec('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Aliens vs Predator: Requiem
Ender's Game


In [17]:
import pickle
import gzip

# Save compressed pickle files
with gzip.open('movies.pkl.gz', 'wb') as f:
    pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

with gzip.open('similar.pkl.gz', 'wb') as f:
    pickle.dump(similar, f, protocol=pickle.HIGHEST_PROTOCOL)
